# PC1 resuelta: Compania de Minas Buenaventura S.A.A.

**Curso:** Topicos de Finanzas Avanzadas - UPAO
**Empresa:** Compania de Minas Buenaventura S.A.A. (`BVN`)


## Indicaciones de entrega

El examen cubre las semanas 1 a 3. Ejecuta `Restart & Run All`, revisa que todas las tablas tengan datos, exporta el notebook a un solo PDF y subelo a Canvas antes del domingo 20 de setiembre a las 11:59 pm. Como respaldo, haz commit del notebook en tu fork.

## Parte 1: los conceptos con mis palabras

### 1.1
El valor intrinseco de Buenaventura depende de supuestos sobre sus flujos futuros, riesgo y crecimiento, mientras que el precio de mercado tambien incorpora expectativas y cambios de animo. Por eso una diferencia entre ambos sugiere una posible oportunidad, pero no dice cuanto ganare: el mercado puede tardar en reconocerla y mis supuestos pueden estar equivocados.

### 1.2
La tasa de descuento de Buenaventura se construye porque combina el riesgo del negocio minero, el financiamiento y el mercado donde opera,, no existe una tasa unica publicada que represente exactamente a la empresa. Dos decisiones fueron la tasa libre de riesgo y la prima de riesgo del mercado, y tambien tuve que decidir si incorporaba una prima por riesgo pais.

### 1.3
La utilidad neta se calcula con criterios contables y puede incluir ventas aun no cobradas o gastos sin salida inmediata de caja. En Buenaventura, por ejemplo, la caja puede caer si aumenta el capital de trabajo o se realizan inversiones mineras, aunque la utilidad reportada haya crecido.

### 1.4
Lo que Buenaventura podria repartir a sus accionistas se llama FCFE: es el flujo disponible despues de inversiones y financiamiento. Lo que efectivamente decide pagar se llama dividendo; puede ser menor que el FCFE porque la empresa puede retener caja para invertir, amortizar deuda o mantener liquidez.

## Parte 2: encuentra los errores

Datos: EBIT = 50, t = 30%, depreciacion = 10, capex = 15, incremento de capital de trabajo = 5, intereses = 8, endeudamiento neto = 0, Ke = 12% y WACC = 9%.

### Los cuatro errores conceptuales

1. En el FCFF se sumo el interes completo, pero debe sumarse el interes despues de impuestos, Interes x (1-t), porque el interes genera un ahorro fiscal.
2. Se obtuvo FCFE restando el interes completo al FCFF,  lo correcto es 
    FCFE = FCFF - Interes x (1-t) + endeudamiento neto.
3. Se desconto el FCFE al WACC, pero el FCFE pertenece a los accionistas y debe descontarse al costo del equity, 
Ke.
4. Se uso un crecimiento historico de 7% como crecimiento perpetuo sin justificarlo; para este ejercicio el enunciado pide 
g = 3%, y en una perpetuidad g debe ser menor que la tasa de descuento.

In [1]:
# Recalculo de la Parte 2
EBIT = 50
tax = 0.30
depreciacion = 10
capex = 15
incremento_wc = 5
interes = 8
endeudamiento_neto = 0
ke = 0.12
wacc = 0.09
g = 0.03

fcff = EBIT * (1 - tax) + depreciacion - capex - incremento_wc
fcfe = fcff - interes * (1 - tax) + endeudamiento_neto
valor_equity = fcfe * (1 + g) / (ke - g)

print(f'FCFF corregido = {fcff:.2f}')
print(f'FCFE corregido = {fcfe:.2f}')
print(f'Valor del equity = {valor_equity:.2f}')

FCFF corregido = 25.00
FCFE corregido = 19.40
Valor del equity = 222.02


### Resultado de la correccion

FCFF = 50(1 - 0.30) + 10 - 15 - 5 = 25.

FCFE = 25 - 8(1 - 0.30) + 0 = 19.4.

Como el FCFE del siguiente periodo es 19.4 x 1.03 = 19.982 y se descuenta al Ke = 12%, el valor correcto del equity es `19.982 / (0.12 - 0.03) = 222.02`.

## Parte 3: mi empresa, Buenaventura

La siguiente celda descarga los estados financieros publicados por 
yfinance. Las cifras pueden cambiar si la fuente actualiza sus datos, por lo que debo conservar la fecha de consulta y revisar que los signos tengan sentido.

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import yfinance as yf

sys.path.insert(0, os.path.abspath('..'))
from utils.finanzas import fcff_desde_ebit, fcfe_desde_fcff

TICKER = 'BVN'
empresa = yf.Ticker(TICKER)
est = empresa.income_stmt
bal = empresa.balance_sheet
cf = empresa.cashflow

def fila(df, *nombres, default=None):
    for nombre in nombres:
        if nombre in df.index:
            return df.loc[nombre]
    if default is not None:
        return pd.Series(default, index=df.columns, dtype=float)
    raise KeyError(f'No se encontro ninguna fila: {nombres}')

def numero_positivo(serie):
    return pd.to_numeric(serie, errors='coerce').abs()

ebit = fila(est, 'EBIT', 'Operating Income')
pretax = fila(est, 'Pretax Income', 'Income Before Tax')
tax_provision = numero_positivo(fila(est, 'Tax Provision'))
try:
    depreciacion = numero_positivo(fila(cf, 'Depreciation And Amortization', 'Depreciation Amortization Depletion'))
except KeyError:
    depreciacion = numero_positivo(fila(est, 'Reconciled Depreciation', 'Depreciation Amortization Depletion Income Statement', 'Depreciation And Amortization In Income Statement'))
capex = numero_positivo(fila(cf, 'Capital Expenditure', 'Capital Expenditure Reported'))
interes = numero_positivo(fila(est, 'Interest Expense', 'Interest Expense Non Operating', default=0))

# El cambio de capital de trabajo consume caja cuando es positivo.
cambio_wc = -pd.to_numeric(fila(cf, 'Change In Working Capital', default=0), errors='coerce')

t_efectiva_serie = (tax_provision / pretax.replace(0, np.nan)).clip(lower=0, upper=0.50)
datos = pd.DataFrame({
    'EBIT': ebit,
    'tasa efectiva': t_efectiva_serie,
    'Depreciacion': depreciacion,
    'Capex': capex,
    'Incremento WC': cambio_wc,
    'Interes': interes,
}).dropna(subset=['EBIT', 'Depreciacion', 'Capex'])

datos['tasa efectiva'] = datos['tasa efectiva'].fillna(0.295)
datos['Incremento WC'] = datos['Incremento WC'].fillna(0)
datos['Interes'] = datos['Interes'].fillna(0)
datos = datos.sort_index(ascending=False).head(4)

for columna in ['EBIT', 'Depreciacion', 'Capex', 'Incremento WC', 'Interes']:
    datos[columna] = datos[columna] / 1e6

datos.index = pd.to_datetime(datos.index).year
datos['FCFF por EBIT'] = [
    fcff_desde_ebit(fila_anio['EBIT'], fila_anio['tasa efectiva'], fila_anio['Depreciacion'],
                   fila_anio['Capex'], fila_anio['Incremento WC'])
    for _, fila_anio in datos.iterrows()
]

print(empresa.info.get('longName', TICKER))
display(datos.round(2))

Compañía de Minas Buenaventura S.A.A.


,EBIT,tasa efectiva,Depreciacion,Capex,Incremento WC,Interes,FCFF por EBIT
2025,1025.70,0.13,2.41,473.01,0.00,58.39,419.16
2024,616.50,0.27,2.43,337.74,0.00,43.05,113.30
2023,180.54,0.50,2.00,238.67,-4.95,98.01,-141.45
2022,168.52,0.00,2.46,151.97,0.00,44.09,18.95


### 3.1 FCFF del ultimo ano disponible

Para el ultimo ano de la tabla, el FCFF se calcula como `EBIT(1-t) + depreciacion - capex - incremento de capital de trabajo`. Esta ruta empieza en la utilidad operativa y por eso no resta intereses: el FCFF representa el efectivo disponible para todos los financiadores.

In [3]:
ultimo = datos.iloc[0]
print('Ano:', datos.index[0])
print(f"EBIT: USD {ultimo['EBIT']:,.2f} millones")
print(f"Tasa efectiva: {ultimo['tasa efectiva']:.2%}")
print(f"Depreciacion: USD {ultimo['Depreciacion']:,.2f} millones")
print(f"Capex: USD {ultimo['Capex']:,.2f} millones")
print(f"Incremento de WC: USD {ultimo['Incremento WC']:,.2f} millones")
print(f"FCFF: USD {ultimo['FCFF por EBIT']:,.2f} millones")

Ano: 2025
EBIT: USD 1,025.70 millones
Tasa efectiva: 13.25%
Depreciacion: USD 2.41 millones
Capex: USD 473.01 millones
Incremento de WC: USD 0.00 millones
FCFF: USD 419.16 millones


### 3.2 FCFF o FCFE para valorar el equity

Usaria **FCFF** y lo descontaria al WACC porque Buenaventura es una empresa minera y su deuda puede cambiar con el ciclo de inversiones, los precios de los metales y sus proyectos. El FCFF permite valorar primero toda la operacion y luego restar la deuda neta; ademas, si el FCFE resulta negativo o muy volatil, es menos estable como base directa para valorar el equity.

La siguiente celda verifica el FCFE aproximado del ultimo ano usando el interes despues de impuestos y endeudamiento neto igual a cero como aproximacion cuando no se dispone de toda la serie de financiamiento.

In [4]:
def valor_balance(df, *nombres):
    for nombre in nombres:
        if nombre in df.index:
            valor = pd.to_numeric(df.loc[nombre].iloc[0], errors='coerce')
            return float(valor) / 1e6
    return np.nan

deuda = valor_balance(bal, 'Total Debt')
caja = valor_balance(bal, 'Cash And Cash Equivalents',
                     'Cash Cash Equivalents And Short Term Investments')
deuda_neta = deuda - caja
endeudamiento_neto_aprox = 0.0
fcfe_aprox = fcfe_desde_fcff(
    ultimo['FCFF por EBIT'],
    ultimo['Interes'],
    ultimo['tasa efectiva'],
    endeudamiento_neto_aprox,
)

print(f'Deuda total: USD {deuda:,.2f} millones')
print(f'Caja: USD {caja:,.2f} millones')
print(f'Deuda neta: USD {deuda_neta:,.2f} millones')
print(f'FCFE aproximado: USD {fcfe_aprox:,.2f} millones')
print('Lectura: un FCFE positivo no elimina la volatilidad de la deuda; por eso la valoracion principal usa FCFF.')

Deuda total: USD 8.93 millones
Caja: USD 529.84 millones
Deuda neta: USD -520.91 millones
FCFE aproximado: USD 368.51 millones
Lectura: un FCFE positivo no elimina la volatilidad de la deuda; por eso la valoracion principal usa FCFF.


### 3.3 El interes y el FCFF

El interes que pago Buenaventura corresponde a sus acreedores o financistas, como bancos y tenedores de deuda. No se resta al calcular el FCFF porque el FCFF representa el efectivo disponible para todos los financiadores y se calcula antes de decidir cuanto corresponde a acreedores y accionistas; si parto del EBIT, los intereses aun no han sido restados. El efecto fiscal de la deuda se incorpora en el WACC, o se ajusta cuando se parte de utilidad neta.

## Parte 4: tu y tu IA

### 4.1 Prompt mas util

Actua como tutor de finanzas corporativas. Ayudame a verificar mi calculo de FCFF de Buenaventura usando la ruta del EBIT. Explica cada insumo, revisa los signos de depreciacion, capex y capital de trabajo, y no inventes datos: indica que debo comprobar en los estados financieros de yfinance.

### 4.2 Que tuve que verificar o corregir

La IA puede entregar una formula correcta pero aplicar mal el signo del cambio en capital de trabajo o confundir FCFF con FCFE. Lo verifique comparando la formula con el mock, revisando las etiquetas de los estados financieros de BVN y ejecutando el notebook desde cero. Tambien corregi el ejercicio de Andina: el interes debe ajustarse por impuestos para pasar de FCFF a FCFE y el FCFE debe descontarse al Ke, no al WACC.

